### Prepare repo and dependencies

In [ ]:
!git clone https://github.com/pohaoc2/hover_net.git

In [ ]:
from IPython.display import clear_output
%cd /content/hover_net
!pip install --upgrade pip setuptools wheel
!sed -i 's/numpy==1.23.5/numpy>=1.26.0,<2.0.0/g' /content/hover_net/requirements.txt
!pip install "numpy>=1.26.0,<2.0.0"
!pip install -r /content/hover_net/requirements.txt
!pip install openslide-python==1.1.2


### Download data and checkpoints

In [ ]:
import gdown
# checkpoint
file_id = "1C8EhGI95--F1KdxoMq4zyhkfrkJLHJuK"
output = "hovernet_original_consep_notype_pytorch.tar.gz"
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

# data
guidance_scale = 1
!mkdir seg_outputs_w{guidance_scale}
file_id = "1nO0AdF_IP18O58L5tUHdk0ko5wIXJJq8" if guidance_scale == 6 else "1Ba-isjTq24aHkzPMoOouXS9ckdTpCYK0"  #https://drive.google.com/file/d/1nO0AdF_IP18O58L5tUHdk0ko5wIXJJq8/view?usp=sharing # w6
file_id = "1uOmbYjxW5L5uAIj_x2U853FSsO6lQ3G2" # Match dataset
output = f"all_outputs_w{guidance_scale}.zip"
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

In [ ]:
!unzip all_outputs_w{guidance_scale}.zip # w6

### Inference

In [ ]:
%cd /content/hover_net/
!python3 run_infer.py \
         --model_path hovernet_original_consep_notype_pytorch.tar.gz \
         --model_mode original \
         tile \
         --input_dir all_outputs_w{guidance_scale}/generated_images \
         --output_dir seg_outputs_w{guidance_scale}

In [ ]:
!zip -r seg_outputs_w{guidance_scale}.zip seg_outputs_w{guidance_scale}

### Viz

In [ ]:
!pip install --force-reinstall "numpy<2.0.0" scipy matplotlib pandas scikit-learn

In [ ]:
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
%cd hover_net
batch_id = 0
sample_id = 0

# PixCell in/output
original_image_path = f"generated_data/exp_images/{batch_id}_{sample_id}*.png"
mask_path = f"generated_data/masks/{batch_id}_{sample_id}*.png"
generated_image_path = f"generated_data/generated_images/{batch_id}_{sample_id}*.png"

# Hovernet output
mat_path = "seg_outputs/mat/2_0_train_27_013.mat"
mat_data = loadmat(mat_path)
inst_map = mat_data['inst_map']
gen_img = np.array(Image.open(generated_image_path))
overlay = gen_img.copy()
inst_list = np.unique(inst_map)
inst_list = inst_list[inst_list != 0]

for inst_id in inst_list:
    mask = np.array(inst_map == inst_id, np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 1)

fig, ax = plt.subplots(1, 5, figsize=(20, 4))

ax[0].imshow(Image.open(original_image_path))
ax[0].set_title("Original Image")
ax[1].imshow(Image.open(mask_path))
ax[1].set_title("Mask")
ax[2].imshow(gen_img)
ax[2].set_title("Generated Image")
ax[3].imshow(inst_map, cmap='jet')
ax[3].set_title("Instance Map")
ax[4].imshow(overlay)
ax[4].set_title("Overlay (Contours)")
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()
